![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 01: Foundations)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 1A: Python, Colab and API Safety

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Main output</td><td>A safe mini tool system with secure configuration, structured state, tests and reflection</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m01a-overview)
2. [Notebook Setup and Safe Secret Handling](#m01a-setup-secrets)
3. [Structured Workflow State](#m01a-workflow-state)
4. [Safe Tool Design](#m01a-safe-tool)
5. [Tool Registry and Execution Trace](#m01a-tool-registry)
6. [Student Tasks](#m01a-student-tasks)
7. [Submission and Reflection](#m01a-submission)

---


<a id="m01a-overview"></a>

### 1. Overview and Learning Goals

This first practical session establishes the programming and safety foundation for the whole unit. Later sessions will use Flowise, LangChain, LangGraph, retrieval-augmented generation, Hugging Face models and multi-agent frameworks. Those tools look more advanced, but they still depend on a small number of basic engineering habits: configuration should be separated from code, workflow state should be inspectable, tool functions should validate their inputs, and every workflow should be tested with normal and failure cases.

In this notebook, you will build a small but complete workflow that behaves like a simplified agentic system. The workflow receives a structured request, checks which tool should be used, runs a safe Python function, stores the result in a state dictionary, and produces a final answer. The example tool is a calculator, but the design pattern is the same as a later search tool, retrieval tool, document parser or model-evaluation tool.

A major safety theme in this session is API-key handling. Many AI services require API keys. These keys should be treated as private credentials. A public teaching notebook should never contain a real API key. This notebook therefore demonstrates safe patterns using environment variables and `getpass`, while avoiding real service calls.

By the end of this lab, you should be able to explain why API keys should not be hard-coded, load configuration values from outside the notebook, represent simple workflow state with dictionaries, write safe tool-like functions with explicit input validation, avoid unsafe execution patterns such as `eval`, and test a small tool with normal, edge and failure cases.

<div align="center">

<table>
<thead>
<tr><th><strong>Time</strong></th><th><strong>Activity</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">0-10 min</td><td>Overview and motivation</td></tr>
<tr><td align="left">10-25 min</td><td>Notebook environment, API-key safety and workflow state</td></tr>
<tr><td align="left">25-55 min</td><td>Build safe configuration and structured state</td></tr>
<tr><td align="left">55-80 min</td><td>Build a safe calculator tool and tool registry</td></tr>
<tr><td align="left">80-100 min</td><td>Student challenge: add a new tool</td></tr>
<tr><td align="left">100-110 min</td><td>Testing and debugging</td></tr>
<tr><td align="left">110-120 min</td><td>Submission and reflection</td></tr>
</tbody>
</table>

</div>

Recommended background references:

- Python `getpass` documentation: <https://docs.python.org/3/library/getpass.html>
- Python `os` documentation: <https://docs.python.org/3/library/os.html>
- Python `ast` documentation: <https://docs.python.org/3/library/ast.html>
- OWASP Secrets Management Cheat Sheet: <https://cheatsheetseries.owasp.org/cheatsheets/Secrets_Management_Cheat_Sheet.html>
- Google Colab: <https://colab.research.google.com>

<a id="m01a-setup-secrets"></a>

### 2. Notebook Setup and Safe Secret Handling

This notebook uses only the Python standard library. The imports below are intentionally conservative because the first session should work in a fresh Colab runtime without package installation. Later notebooks will install external packages, but this session should remain lightweight.

An API key should be treated like a password. A common mistake is to paste a real key directly into a notebook cell, for example `OPENAI_API_KEY = "sk-..."`. That pattern is unsafe for a public repository and can leak through screenshots, exported notebooks or Git commits. A safer pattern is to load the key from outside the notebook. In Colab, the preferred method is often Colab Secrets. In local development, environment variables or a local `.env` file can be used. This notebook demonstrates the core idea using only standard Python.

In [ ]:
# os gives access to environment variables such as OPENAI_API_KEY or GOOGLE_API_KEY.
# Environment variables are a common way to keep configuration separate from code.
import os

# json is used for pretty-printing dictionaries.
# Agentic workflows often use JSON-like state objects to store intermediate results.
import json

# math provides constants and numeric functions.
# We use it later in the student challenge for the area of a circle.
import math

# operator gives function versions of arithmetic operations.
# For example, operator.add is safer and clearer than evaluating the string "a + b".
import operator

# getpass asks the user to type a value without showing it in normal notebook output.
# This is useful for temporary secret input.
from getpass import getpass

# typing improves readability by making function inputs and outputs explicit.
# It does not change runtime behaviour, but it helps students understand expected data structures.
from typing import Any, Callable, Dict

import sys
print("Python version:", sys.version.split()[0])
print("Notebook setup: OK")

In [ ]:
def load_secret_from_environment(env_name: str, ask_if_missing: bool = False) -> str | None:
    """Load a secret value from an environment variable without printing it.

    The function demonstrates a safe configuration pattern. It keeps secrets
    outside the notebook source code and never prints the secret value.
    """

    # First check whether the value already exists in the environment.
    value = os.environ.get(env_name)
    if value:
        return value

    # If the value is missing and prompting is enabled, ask through getpass.
    # This stores the value only in the current runtime environment.
    if ask_if_missing:
        value = getpass(f"Enter {env_name}: ")
        os.environ[env_name] = value
        return value

    # If the key is unavailable and prompting is not requested, return None.
    return None

In [ ]:
# Demonstration with a dummy value only. This is not a real API key.
os.environ["DUMMY_API_KEY"] = "DUMMY_VALUE_FOR_TESTING_ONLY"

dummy_key = load_secret_from_environment("DUMMY_API_KEY", ask_if_missing=False)

# Good practice: print only whether a key exists, not the key itself.
print("DUMMY_API_KEY exists:", dummy_key is not None)

# Clean up the dummy value after the demonstration.
del os.environ["DUMMY_API_KEY"]

The expected result is a Boolean message indicating that the dummy key exists. The key value itself should not appear in the output. This is the behaviour we want later when a notebook checks for a real model-provider key. If a key is missing, the notebook should explain what needs to be configured; it should not include the key in source code.

<a id="m01a-workflow-state"></a>

### 3. Structured Workflow State

Agentic workflows often need temporary memory. This does not necessarily mean long-term user memory. It can simply mean a state object that records what has happened inside a workflow. For example, a later LangGraph workflow may store the user's question, retrieved documents, selected tool, draft answer, reviewer feedback and final answer.

Here we use a Python dictionary to represent the same idea in a simpler form. The state dictionary is deliberately explicit. Each field has a clear purpose. This makes the workflow easier to inspect, debug and test.

```mermaid
flowchart LR
    A[User request] --> B[Structured state]
    B --> C[Tool selection]
    C --> D[Input validation]
    D --> E[Safe tool execution]
    E --> F[State update]
    F --> G[Final answer]
```

In [ ]:
# A workflow state is a structured record of the current task.
# We initialise all expected fields at the beginning so that the structure is predictable.
state: Dict[str, Any] = {
    "user_request": "Calculate 12 plus 30",
    "selected_tool": None,
    "tool_input": None,
    "tool_output": None,
    "final_answer": None,
    "errors": []
}

print(json.dumps(state, indent=2))

Notice that this state object is not complicated. Its value is that it records the workflow in a structured way. If the final answer is wrong, you can inspect whether the wrong tool was selected, the wrong input was passed, or the tool itself returned an error. This inspectability is one of the most important habits in agentic AI development.

<a id="m01a-safe-tool"></a>

### 4. Safe Tool Design

A beginner might implement a calculator by accepting a string such as `"12 + 30"` and passing it to `eval`. That is unsafe because `eval` can execute arbitrary Python expressions. In an AI-agent setting, this risk is more serious because tool inputs may be produced by a model or by a user.

Instead, we will build a calculator that accepts a restricted operation name and two numeric arguments. This design is less flexible but much safer. It also makes the tool easier to describe, test and connect to later agent frameworks.

<div align="center">

<table>
<thead>
<tr><th><strong>Design choice</strong></th><th><strong>Decision</strong></th><th><strong>Reason</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Input format</td><td>Structured operation name plus two numbers</td><td>Easier to validate than free-form text</td></tr>
<tr><td align="left">Allowed operations</td><td><code>add</code>, <code>subtract</code>, <code>multiply</code>, <code>divide</code>, <code>power</code></td><td>Keeps the tool scope explicit</td></tr>
<tr><td align="left">Error handling</td><td>Return a structured dictionary</td><td>Avoids crashing the workflow</td></tr>
<tr><td align="left">Unsafe execution</td><td>Do not use <code>eval</code> or <code>exec</code></td><td>Prevents arbitrary code execution</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Map operation names to safe Python functions.
# This is safer than evaluating a text expression.
ALLOWED_OPERATIONS: Dict[str, Callable[[float, float], float]] = {
    "add": operator.add,
    "subtract": operator.sub,
    "multiply": operator.mul,
    "divide": operator.truediv,
    "power": operator.pow,
}

def safe_calculator(operation: str, a: Any, b: Any) -> Dict[str, Any]:
    """Run a restricted calculator operation safely.

    Returns a structured dictionary with:
    - ok: whether the operation succeeded
    - error: error message if the operation failed
    - result: numeric result if the operation succeeded
    """

    # Reject operations that are not explicitly allowed.
    if operation not in ALLOWED_OPERATIONS:
        return {"ok": False, "error": f"Unsupported operation: {operation}", "result": None}

    # Reject non-numeric input. Strict validation is easier to reason about.
    if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
        return {"ok": False, "error": "Inputs must be numeric values of type int or float.", "result": None}

    # Handle a known mathematical failure case before running the operation.
    if operation == "divide" and b == 0:
        return {"ok": False, "error": "Division by zero is not allowed.", "result": None}

    # Run the selected safe function after validation.
    result = ALLOWED_OPERATIONS[operation](float(a), float(b))
    return {"ok": True, "error": None, "result": result}

In [ ]:
print("Normal case:")
print(safe_calculator("add", 12, 30))

print("\nFailure case:")
print(safe_calculator("divide", 10, 0))

The normal case should return a successful result. The failure case should not crash the notebook; it should return a controlled error message. This is important because larger workflows should be able to record a tool error and continue with a useful explanation.

<a id="m01a-tool-registry"></a>

### 5. Tool Registry and Execution Trace

A tool registry maps tool names to functions. This idea appears in many agent frameworks. The framework needs to know which tools exist, what they are called, and how to invoke them. In this notebook, the registry has only one tool. Later, the same pattern can support web search, document retrieval, summarisation or model evaluation.

A registry is also useful for safety. Instead of allowing arbitrary Python functions to be called, the workflow can only call functions that are explicitly registered.

In [ ]:
ToolFunction = Callable[..., Dict[str, Any]]

# The registry exposes only tools that we intentionally allow.
tool_registry: Dict[str, ToolFunction] = {
    "calculator": safe_calculator
}

print("Registered tools:", list(tool_registry.keys()))

In [ ]:
# Select the calculator tool and prepare structured input.
state["selected_tool"] = "calculator"
state["tool_input"] = {"operation": "add", "a": 12, "b": 30}

tool_name = state["selected_tool"]
tool_input = state["tool_input"]

# Check whether the requested tool exists in the registry.
if tool_name in tool_registry:
    tool_output = tool_registry[tool_name](**tool_input)
else:
    tool_output = {"ok": False, "error": f"Unknown tool: {tool_name}", "result": None}

# Store raw tool output and prepare final answer.
state["tool_output"] = tool_output

if tool_output["ok"]:
    state["final_answer"] = f"The result is {tool_output['result']}."
else:
    state["errors"].append(tool_output["error"])
    state["final_answer"] = "The tool could not complete the request."

print(json.dumps(state, indent=2))

Inspect the printed state. You should be able to trace the whole workflow from the original request to the final answer. This traceability is the main reason for using structured state.

<a id="m01a-student-tasks"></a>

### 6. Student Tasks

This section contains the required student work for this session. The earlier sections introduced the safety principles and guided implementation. Here you should actively test the existing code, complete the missing tool, and demonstrate that your implementation is safe and inspectable.

The key learning outcome is not simply to make a function return a number. The purpose is to practise the same tool-development discipline used in agentic AI systems: define the allowed input, reject unsafe or invalid input, return structured output, register the tool explicitly, and test both success and failure behaviour.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Task</strong></th>
<th><strong>What you need to do</strong></th>
<th><strong>Why it matters</strong></th>
<th><strong>Expected evidence</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Task 1</td>
<td>Run the calculator tests and confirm that normal, edge and failure cases behave correctly.</td>
<td>Testing confirms that a tool is reliable beyond one successful example.</td>
<td>Output showing that all calculator tests passed.</td>
</tr>
<tr>
<td align="left">Task 2</td>
<td>Read the unsafe-pattern example and explain why user-controlled text should not be passed into <code>eval</code>.</td>
<td>Unsafe execution is a major risk when LLMs or users generate tool inputs.</td>
<td>A short explanation in your reflection.</td>
</tr>
<tr>
<td align="left">Task 3</td>
<td>Complete the <code>circle_area</code> function using the same structured-output style as <code>safe_calculator</code>.</td>
<td>New tools should follow the same interface pattern so that the workflow can call them consistently.</td>
<td>A working function that returns <code>ok</code>, <code>error</code> and <code>result</code>.</td>
</tr>
<tr>
<td align="left">Task 4</td>
<td>Register <code>circle_area</code> in the tool registry.</td>
<td>An agentic workflow should only call explicitly registered tools.</td>
<td>The registry output includes both <code>calculator</code> and <code>circle_area</code>.</td>
</tr>
<tr>
<td align="left">Task 5</td>
<td>Run normal, edge and failure tests for <code>circle_area</code>.</td>
<td>The tests show whether the tool handles valid and invalid input safely.</td>
<td>Test output showing that the implemented tool behaves correctly.</td>
</tr>
</tbody>
</table>

</div>

For the `circle_area` tool, use the formula `area = pi * radius * radius`. The tool should accept only numeric radius values. It should reject negative values because a circle radius cannot be negative. It should return a structured dictionary rather than only printing a result, because later workflow components need to inspect whether the tool succeeded.

Recommended behaviour:

<div align="center">

<table>
<thead>
<tr>
<th><strong>Input case</strong></th>
<th><strong>Example</strong></th>
<th><strong>Expected behaviour</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Normal case</td>
<td><code>radius = 1</code></td>
<td>Return <code>ok=True</code> and area approximately equal to <code>math.pi</code>.</td>
</tr>
<tr>
<td align="left">Edge case</td>
<td><code>radius = 0</code></td>
<td>Return <code>ok=True</code> and area equal to <code>0</code>.</td>
</tr>
<tr>
<td align="left">Failure case</td>
<td><code>radius = -1</code></td>
<td>Return <code>ok=False</code> with a clear error message.</td>
</tr>
<tr>
<td align="left">Failure case</td>
<td><code>radius = "large"</code></td>
<td>Return <code>ok=False</code> because the input is not numeric.</td>
</tr>
</tbody>
</table>

</div>

If you need more detail about the Python features used in this task, refer to the Python documentation for functions, dictionaries and exceptions: <https://docs.python.org/3/tutorial/controlflow.html#defining-functions> and <https://docs.python.org/3/tutorial/datastructures.html#dictionaries>.


In [ ]:
# Normal cases: these should succeed.
assert safe_calculator("add", 2, 3)["result"] == 5.0
assert safe_calculator("subtract", 10, 4)["result"] == 6.0
assert safe_calculator("multiply", 6, 7)["result"] == 42.0

# Edge cases: these are valid but check boundary-like behaviour.
assert safe_calculator("power", 2, 0)["result"] == 1.0
assert safe_calculator("divide", 5, 2)["result"] == 2.5

# Failure cases: these should be rejected without crashing.
assert safe_calculator("divide", 5, 0)["ok"] is False
assert safe_calculator("unknown", 5, 2)["ok"] is False
assert safe_calculator("add", "5", 2)["ok"] is False

print("All calculator tests passed.")

The next example shows why the notebook avoids `eval`. The string below looks like ordinary text, but if a program blindly passes user text into `eval`, Python may interpret it as executable code. This notebook does not execute the string. It only displays it as a warning example.

In [ ]:
malicious_like_input = "__import__('os').environ"

print("Unsafe user-controlled input example:")
print(malicious_like_input)

safe_result = safe_calculator("add", 2, 3)
print("Safe calculator result:", safe_result)

Now add a new tool called `circle_area`. The tool should accept one input, `radius`, and return the area of a circle. This challenge is not mainly about geometry. It is about adding a tool safely to the same design pattern.

The tool should accept structured input, validate the input type, reject invalid values, return a dictionary with `ok`, `error` and `result`, and include tests for normal, edge and failure cases.

The formula is:

```text
area = pi * radius * radius
```

In [ ]:
def circle_area(radius: Any) -> Dict[str, Any]:
    """Calculate the area of a circle safely.

    This function is the student task for this notebook. It should follow the
    same structured-output convention as safe_calculator.

    Parameters
    ----------
    radius:
        Radius of the circle. It must be an int or float and must not be negative.

    Returns
    -------
    dict
        A dictionary with:
        - ok: True if the operation succeeded, otherwise False
        - error: None if successful, otherwise a clear error message
        - result: area of the circle if successful, otherwise None

    Implementation requirements
    ---------------------------
    1. Validate the input type before doing the calculation.
    2. Reject negative radius values.
    3. Use math.pi for the constant pi.
    4. Return structured output. Do not only print the result.
    """

    # Step 1: check that radius is numeric.
    # We accept int and float because they are the standard numeric types
    # needed for this simple geometry calculation.
    #
    # If the input is a string such as "large", the function should reject it.
    # Do not automatically convert strings in this first lab, because strict
    # validation is easier to inspect and safer for tool design.
    #
    # TODO: implement this validation.
    # Example failure return:
    # return {"ok": False, "error": "Radius must be numeric.", "result": None}

    # Step 2: check that radius is not negative.
    # A radius of 0 is valid and should return area 0.
    # A negative radius is invalid and should be rejected with a clear error.
    #
    # TODO: implement this validation.
    # Example failure return:
    # return {"ok": False, "error": "Radius must not be negative.", "result": None}

    # Step 3: compute the area.
    # Use math.pi rather than typing an approximate value such as 3.14.
    #
    # TODO: compute the result.
    # area = ...

    # Step 4: return the structured success result.
    #
    # TODO: return:
    # return {"ok": True, "error": None, "result": area}

    raise NotImplementedError("Complete the circle_area function.")


# After implementing circle_area, uncomment the following two lines.
# The registry step is required because the workflow should only call explicitly allowed tools.
# tool_registry["circle_area"] = circle_area
# print("Registered tools:", list(tool_registry.keys()))


In [ ]:
# Tests for the student task.
# Uncomment these tests after implementing circle_area.
#
# These tests are deliberately grouped into:
# - normal case: common valid input
# - edge case: boundary-like valid input
# - failure cases: invalid inputs that should be rejected cleanly

# Normal case: radius 1 should produce area pi.
# normal = circle_area(1)
# assert normal["ok"] is True
# assert round(normal["result"], 4) == round(math.pi, 4)

# Edge case: radius 0 is valid and should produce area 0.
# edge = circle_area(0)
# assert edge["ok"] is True
# assert edge["result"] == 0

# Failure case: negative radius should be rejected.
# failure_negative = circle_area(-1)
# assert failure_negative["ok"] is False
# assert failure_negative["result"] is None

# Failure case: non-numeric input should be rejected.
# failure_type = circle_area("large")
# assert failure_type["ok"] is False
# assert failure_type["result"] is None

# print("circle_area tests passed.")


<a id="m01a-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook with the following evidence. The emphasis is on safe workflow design, not only on getting a numeric answer.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Required item</strong></th>
<th><strong>What to submit</strong></th>
<th><strong>Quality check</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Safe configuration</td>
<td>The <code>load_secret_from_environment</code> function and dummy-key demonstration.</td>
<td>The notebook should not print or store any real API key.</td>
</tr>
<tr>
<td align="left">Safe calculator</td>
<td>The implemented <code>safe_calculator</code> and its tests.</td>
<td>Normal, edge and failure tests should pass.</td>
</tr>
<tr>
<td align="left">Tool registry</td>
<td>The registry containing explicitly allowed tools.</td>
<td>The registry should include only intended tools.</td>
</tr>
<tr>
<td align="left">Student tool</td>
<td>The completed <code>circle_area</code> function.</td>
<td>The function should validate input and return structured output.</td>
</tr>
<tr>
<td align="left">Student tests</td>
<td>Tests for <code>circle_area</code>.</td>
<td>Include normal, edge and failure cases.</td>
</tr>
<tr>
<td align="left">Reflection</td>
<td>A short reflection of 150 to 250 words.</td>
<td>The reflection should refer to concrete design decisions in your implementation.</td>
</tr>
</tbody>
</table>

</div>

Your reflection should answer these questions:

1. Why should API keys not be hard-coded in public notebooks?
2. Why is `eval` unsafe for user-controlled tool input?
3. How does dictionary state make the workflow easier to debug?
4. What failure case did you test, and what behaviour did you expect?
5. How does this lab prepare you for later agentic AI workflows?

Use the debugging guide below if your notebook does not behave as expected.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Symptom</strong></th>
<th><strong>Likely cause</strong></th>
<th><strong>How to inspect</strong></th>
<th><strong>Typical fix</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Secret is missing</td>
<td>Environment variable not set</td>
<td>Check whether <code>load_secret_from_environment</code> returns <code>None</code></td>
<td>Use Colab Secrets, environment variables or <code>getpass</code></td>
</tr>
<tr>
<td align="left">Secret printed accidentally</td>
<td>Code printed the key value</td>
<td>Search notebook outputs for the key</td>
<td>Remove output and rotate the key if real</td>
</tr>
<tr>
<td align="left">Calculator rejects input</td>
<td>Input is not numeric or operation unsupported</td>
<td>Inspect <code>operation</code>, <code>a</code> and <code>b</code></td>
<td>Use allowed operation names and numeric values</td>
</tr>
<tr>
<td align="left">Division fails</td>
<td>Division by zero</td>
<td>Inspect <code>b</code></td>
<td>Keep explicit zero check</td>
</tr>
<tr>
<td align="left">State looks incomplete</td>
<td>Some fields were not updated</td>
<td>Print <code>state</code> as JSON</td>
<td>Update the state after every major step</td>
</tr>
<tr>
<td align="left">Challenge tests fail</td>
<td><code>circle_area</code> validation incomplete</td>
<td>Run tests one by one</td>
<td>Check type validation and negative-radius handling</td>
</tr>
</tbody>
</table>

</div>

#### Further Readings


- Python functions: <https://docs.python.org/3/tutorial/controlflow.html#defining-functions>
- Python dictionaries: <https://docs.python.org/3/tutorial/datastructures.html#dictionaries>
- Python `getpass`: <https://docs.python.org/3/library/getpass.html>
- Python `os`: <https://docs.python.org/3/library/os.html>
- OWASP Secrets Management Cheat Sheet: <https://cheatsheetseries.owasp.org/cheatsheets/Secrets_Management_Cheat_Sheet.html>
- Google Colab: <https://colab.research.google.com>
